## Finetuned FinBERT proof of concept

This loads our fine tuned FinBERT model trained with price-volume derived scores and on input 

In [ ]:
pip install iTransformer torch transformers pandas tqdm

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

class FinBERTRegression(nn.Module):
    def __init__(self, model_name="yiyanghkust/finbert-tone"):
        super(FinBERTRegression, self).__init__()
        self.bert = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
        self.regressor = nn.Linear(3, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        regression_score = self.regressor(outputs.logits)
        normalized_score = torch.tanh(regression_score)
        return normalized_score

class AdaptiveWeightedHuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super(AdaptiveWeightedHuberLoss, self).__init__()
        self.delta = delta

    def forward(self, predictions, targets, weights=None):
        error = predictions - targets
        abs_error = torch.abs(error)
        quadratic = torch.minimum(abs_error, torch.tensor(self.delta))
        linear = abs_error - quadratic
        loss = 0.5 * quadratic ** 2 + self.delta * linear
        if weights is not None:
            loss = loss * weights
        return loss.mean()



model = FinBERTRegression()
model.load_state_dict(torch.load('./fine-tuned-finbert.pth'))
model.eval()

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")


In [ ]:
import torch
import pandas as pd
from tqdm.notebook import tqdm # Import tqdm

text = "The company's revenue exceeded expectations, leading to a positive outlook for the next quarter."


inputs_inf = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=256
    ).to(device)

with torch.no_grad():
    sentiment = model(
        input_ids=inputs_inf["input_ids"],
        attention_mask=inputs_inf["attention_mask"]
    ).item()

print("Input text:", text)
print(f"Sentiment score: {sentiment}")